# Neurality: Creator Repetition Limiter & Penalty Systems
This notebook implements and benchmarks the `recent_creator_penalty` system to enforce creator diversity and avoid repeating the same creators in the user feed.

In [1]:
# ==================================================
# NOTEBOOK VALIDATION & DEPENDENCY VERIFICATION PIPELINE
# ==================================================
import sys
import time
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib
import seaborn as sns
import torch

print(f"[SUCCESS] Jupyter Kernel Python Version: {sys.version}")
print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"scipy: {scipy.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"seaborn: {sns.__version__}")
print(f"torch: {torch.__version__}")
print("[HEALTH CHECK] Conda kernel detection and package imports are 100% stable!")


[SUCCESS] Jupyter Kernel Python Version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
numpy: 2.1.3
pandas: 2.2.3
scipy: 1.15.3
scikit-learn: 1.6.1
matplotlib: 3.10.0
seaborn: 0.13.2
torch: 2.10.0+cpu
[HEALTH CHECK] Conda kernel detection and package imports are 100% stable!


In [2]:
# Penalty Cooldown Implementation
def apply_creator_penalties(candidates, recent_creators, penalty_weight=0.25):
    """
    Applies a soft cooldown to creators appearing in recent history.
    recent_creators: list of creator usernames recently watched.
    """
    penalized_candidates = []
    for cand in candidates:
        c_copy = dict(cand)
        creator = c_copy['creator']
        
        # Calculate how many times creator appears in recent views
        occurrences = recent_creators.count(creator)
        
        penalty = occurrences * penalty_weight
        c_copy['penalty'] = penalty
        c_copy['final_score'] = max(0.0, c_copy['score_collaborative'] - penalty)
        penalized_candidates.append(c_copy)
        
    return penalized_candidates

mock_candidates = [
    {'id': 1, 'creator': 'creator_a', 'score_collaborative': 0.95},
    {'id': 2, 'creator': 'creator_b', 'score_collaborative': 0.90},
    {'id': 3, 'creator': 'creator_a', 'score_collaborative': 0.85},
    {'id': 4, 'creator': 'creator_c', 'score_collaborative': 0.80},
]

# creator_a was recently watched 2 times
recent_history = ['creator_a', 'creator_a', 'creator_b']

res = apply_creator_penalties(mock_candidates, recent_history)
for c in res:
    print(f"Creator: {c['creator']:10s} | Base Score: {c['score_collaborative']:.2f} | Penalty: {c['penalty']:.2f} | Final: {c['final_score']:.2f}")

Creator: creator_a  | Base Score: 0.95 | Penalty: 0.50 | Final: 0.45
Creator: creator_b  | Base Score: 0.90 | Penalty: 0.25 | Final: 0.65
Creator: creator_a  | Base Score: 0.85 | Penalty: 0.50 | Final: 0.35
Creator: creator_c  | Base Score: 0.80 | Penalty: 0.00 | Final: 0.80


In [3]:
# Gini Coefficient / Creator Diversity Benchmark
def calculate_gini(recommendations):
    """
    Measures concentration/repetition of creators in feed.
    0 = perfect diversity, 1 = absolute concentration (single creator)
    """
    counts = pd.Series([r['creator'] for r in recommendations]).value_counts().values
    n = len(counts)
    if n <= 1:
        return 1.0
    sorted_counts = np.sort(counts)
    index = np.arange(1, n + 1)
    return (np.sum((2 * index - n  - 1) * sorted_counts)) / (n * np.sum(sorted_counts))

# Generate a large candidate pool and run feed ranking
np.random.seed(42)
creators_pool = [f"creator_{i}" for i in range(1, 10)]
candidates = [
    {'creator': np.random.choice(creators_pool), 'score_collaborative': np.random.uniform(0.5, 1.0)}
    for _ in range(100)
]

# Simulate recommendations WITH vs WITHOUT creator penalties
recent_history = ['creator_1', 'creator_1', 'creator_1', 'creator_2', 'creator_2']

candidates_no_penalty = sorted(candidates, key=lambda x: x['score_collaborative'], reverse=True)[:10]
gini_no_penalty = calculate_gini(candidates_no_penalty)

penalized = apply_creator_penalties(candidates, recent_history, penalty_weight=0.3)
candidates_with_penalty = sorted(penalized, key=lambda x: x['final_score'], reverse=True)[:10]
gini_with_penalty = calculate_gini(candidates_with_penalty)

print(f"Creator Repetition concentration (Gini Index) WITHOUT penalty: {gini_no_penalty:.3f}")
print(f"Creator Repetition concentration (Gini Index) WITH penalty:    {gini_with_penalty:.3f}")
assert gini_with_penalty <= gini_no_penalty, "Penalty should decrease repetition and concentration!"

Creator Repetition concentration (Gini Index) WITHOUT penalty: 0.240
Creator Repetition concentration (Gini Index) WITH penalty:    0.250


AssertionError: Penalty should decrease repetition and concentration!